# Generate environments for training

This notebooks explains how you can either use our example curriculum for your training or generate environments yourself.

## Loading competition environments from ecml2026-starterkit (example curriculum)

In [ ]:
from flatland.envs.persistence import RailEnvPersister
from flatland.utils.rendertools import RenderTool
import PIL

In [ ]:
!git clone https://github.com/flatland-association/ecml2026-starterkit -b use-competition-infrastructure-for-top-level-readme

In [ ]:
import sys

sys.path.insert(0, "ecml2026-starterkit")

In [ ]:
!unzip ecml2026-starterkit/reinforcement_learning/curriculum/example_curriculum.zip -d ecml2026-starterkit/reinforcement_learning/curriculum/

In [ ]:
env, _ = RailEnvPersister.load_new("ecml2026-starterkit/reinforcement_learning/curriculum/00_scene_1_ll-2_a-1.pkl")

The parameters in the provided example environments are described in the [ecml2026-starterkit](https://github.com/flatland-association/ecml2026-starterkit/blob/main/reinforcement_learning/curriculum/CURRICULUM.md). 

In [ ]:
env_renderer = RenderTool(env)
image = env_renderer.render_env(show=False, show_observations=False, show_predictions=False, return_image=True)
display(PIL.Image.fromarray(image))

## Deep-Dive Line Generation

In [ ]:
agent = env.agents[0]
agent

Lines consist of a sequence of "flexible waypoints" (a set of routing alternatives). The first waypoint is always unique and the last is a waypoint whose direction is `None` (meaning the cell can to be reached from any direction). Here's an example of such a simple line:

In [ ]:
agent.waypoints

However, lines can consist of more than source and target, namely, intermediate stops. For intermediate stops, it does not matter which of the cells belonging to a given station the agent stops at. This provides flexibility when routing the agents. 

Note: stops at intermediate stops are not enforced, i.e. an agent does not have to stop at or can even omit intermediate waypoints all together, however, it is penalized accordingly.

Sampling lines is done by randomly picking stations. For intermediate stops, a list of waypoints, i.e. all cells belonging to the station, is provided.

## Deep-Dive Schedules

The timetable consist of time windows for each waypoint, an earliest arrival and a latest departure time. For the source (first waypoint), latest arrival is undefined, for the final target (last waypoint), earliest departure is undefined:

In [ ]:
agent.waypoints_earliest_departure

In [ ]:
agent.waypoints_latest_arrival

Timetables are generated for each agent acoording to their lines. For each line the shortest path connecting the waypoints is calculated (only for one cell for intermediate stops). Multiplying the shortest time, which takes into account the agents `max_speed`, with a travel factor (>1) provides the time window (latest arrival, earliest departure) for the selected waypoints. 

The environment does not keep track of intermediate waypoints, i.e. an agent does not have to stop at or can even omit intermediate waypoints all together. The environment only enforces that an agent cannot enter Flatland before earliest departure defined for the agent's initial waypoint. However, the reward-function evaluates whether trains have passed and stopped in the requested time windows and will penalize otherwise.

## Generate new environments

Using the [ecml2026-starterkit](https://github.com/flatland-association/ecml2026-starterkit/blob/main/reinforcement_learning/curriculum/CURRICULUM.md), new environments can be generated. From within the starterkit, use:

In [ ]:
from reinforcement_learning.create_curriculum_envs import create_curriculum_env


env = create_curriculum_env(
    scenario_path = "./sampling/level_0_scenario_1.pkl",  # using this as input provides the infrastructure map from the competition, all other parameters can be overridden, e.g. scene (stations), number of agents and line length 
    scene = "scene_1",  # choose from: "scene_1", "scene_2", "scene_3", "scene_4", "scene_5"
    n_agents_range = None,  # uses the number given in the .pkl as default, can be set to a tuple (min_n_agents, max_n_agents) to sample a random number of agents from the given range
    line_length = 2,  # default is 2, has to be >= 2
)